In [4]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

sys.path.append(str(Path.cwd().parent / "src"))
from airfoil.data import load_data
from airfoil.numerics import solve_lstsq, solve_normal_equations, solve_ridge

df = load_data()
X_in = df.drop(columns="spl_db").to_numpy()   # the 5 inputs
y = df["spl_db"].to_numpy()                    # the target

In [5]:
poly = PolynomialFeatures(degree=3)            # all products up to power 3

X_raw = poly.fit_transform(X_in)                                  # raw inputs
X_std = poly.fit_transform(StandardScaler().fit_transform(X_in))  # scaled first

print("Number of feature columns:", X_raw.shape[1])

Number of feature columns: 56


In [6]:
def rmse(X, beta):
    """Training RMSE in dB for coefficients beta."""
    return np.sqrt(np.mean((X @ beta - y) ** 2))

rows = []
for name, X in [("Raw", X_raw), ("Standardised", X_std)]:
    cond_X = np.linalg.cond(X)
    cond_XtX = np.linalg.cond(X.T @ X)
    rows.append({"Features": name, "Solver": "Normal equations",
                 "cond(X)": cond_X, "cond(matrix solved)": cond_XtX,
                 "RMSE (dB)": rmse(X, solve_normal_equations(X, y))})
    rows.append({"Features": name, "Solver": "lstsq (SVD)",
                 "cond(X)": cond_X, "cond(matrix solved)": cond_X,
                 "RMSE (dB)": rmse(X, solve_lstsq(X, y))})

alpha = 1.0
A_ridge = X_std.T @ X_std + alpha * np.eye(X_std.shape[1])
rows.append({"Features": "Standardised", "Solver": f"Ridge (alpha={alpha})",
             "cond(X)": np.linalg.cond(X_std),
             "cond(matrix solved)": np.linalg.cond(A_ridge),
             "RMSE (dB)": rmse(X_std, solve_ridge(X_std, y, alpha))})

results = pd.DataFrame(rows)
results.style.format({"cond(X)": "{:.2e}", "cond(matrix solved)": "{:.2e}",
                      "RMSE (dB)": "{:.3f}"})

,Features,Solver,cond(X),cond(matrix solved),RMSE (dB)
0,Raw,Normal equations,4.75e+18,2.11e+37,3.382
1,Raw,lstsq (SVD),4.75e+18,4.75e+18,3.878
2,Standardised,Normal equations,2.00e+03,4.01e+06,3.382
3,Standardised,lstsq (SVD),2.00e+03,2.00e+03,3.382
4,Standardised,Ridge (alpha=1.0),2.00e+03,2.70e+05,3.413


## Conditioning of the polynomial regression

I fitted a degree-3 polynomial model two ways, once on the raw inputs and once on standardised inputs and compared three solvers.

**What I found**
- **The normal equations square the condition number.** With standardised inputs, cond(X) = 2.00 × 10³ but cond(XᵀX) = 4.01 × 10⁶, which is exactly (2.00 × 10³)².
- **Raw inputs make the problem numerically singular.** Polynomial columns range from about 10⁻¹³ (delta star³) to 8 × 10¹² (frequency³), so cond(X) = 4.75 × 10¹⁸. That's beyond what a computer's 16 significant digits can handle.
- **This explains the odd raw-input results.** lstsq recognised the matrix as singular and ignored the directions it couldn't resolve, so it fitted about 0.5 dB worse. The normal equations matched the best RMSE, but with a condition number of 10³⁷ their coefficients are dominated by rounding error. The fit looks fine, but the numbers behind it can't be trusted.
- **Standardising fixes it.** It cut cond(X) by about 15 orders of magnitude, and then all three solvers agreed on 3.382 dB.
- **Ridge improves stability cheaply.** It lowered the condition number from 4.01 × 10⁶ to 2.70 × 10⁵ for only 0.03 dB more training error.

**What I learned:** a good RMSE doesn't prove a calculation is numerically sound. I need to check the conditioning too.

**Decision:** I'll always standardise the inputs before building polynomial features and use a stable solver (lstsq or ridge).